In [1]:
# Import libraries

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import matplotlib.pyplot as plt 
import joblib

In [2]:
# Load dataset

X_train = np.load('../data/processed/X_train.npy')
X_test = np.load('../data/processed/X_test.npy')

input_dim = X_train.shape[1]
print(f'Input features: {input_dim}')

Input features: 21


In [3]:
# Define model
input_layer = layers.Input(shape=(input_dim,))

# Encoder
encoded = layers.Dense(16, activation='relu')(input_layer)
encoded = layers.Dense(8, activation='relu')(encoded)      #Latent Space 

# Decoder 
decoded = layers.Dense(16, activation = 'relu')(encoded)

# Using linear because normalized data can be negative
output_layer = layers.Dense(input_dim, activation='linear')(decoded)

# compile
autoencoder = models.Model(inputs=input_layer, outputs= output_layer)
autoencoder.compile(optimizer='adam', loss='mse')


autoencoder.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 21)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 21)             │           357 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 989 (3.86 KB)

 Trainable params: 989 (3.86 KB)

 Non-trainable params: 0 (0.00 B)

In [4]:
# Train

early_stopping = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

print("Starting training...")
history= autoencoder.fit(
    X_train, X_train,   #input is X_train, target is also X_train
    epochs = 50,
    batch_size = 256,
    validation_split = 0.2,
    callbacks = [early_stopping],
    verbose = 1
)

# saving the trained model
autoencoder.save("../models/autoencoder.keras")
print("Model Saved")


Starting training...
Epoch 1/50
5683/5683 ━━━━━━━━━━━━━━━━━━━━ 2s 378us/step - loss: 0.1683 - val_loss: 0.3386
Epoch 2/50
5683/5683 ━━━━━━━━━━━━━━━━━━━━ 2s 352us/step - loss: 0.0466 - val_loss: 0.3049
Epoch 3/50
5683/5683 ━━━━━━━━━━━━━━━━━━━━ 2s 350us/step - loss: 0.0377 - val_loss: 0.2839
Epoch 4/50
5683/5683 ━━━━━━━━━━━━━━━━━━━━ 2s 360us/step - loss: 0.0332 - val_loss: 0.2587
Epoch 5/50
5683/5683 ━━━━━━━━━━━━━━━━━━━━ 2s 364us/step - loss: 0.0305 - val_loss: 0.1951
Epoch 6/50
5683/5683 ━━━━━━━━━━━━━━━━━━━━ 2s 350us/step - loss: 0.0250 - val_loss: 0.1641
Epoch 7/50
5683/5683 ━━━━━━━━━━━━━━━━━━━━ 2s 353us/step - loss: 0.0213 - val_loss: 0.0348
Epoch 8/50
5683/5683 ━━━━━━━━━━━━━━━━━━━━ 2s 353us/step - loss: 0.0191 - val_loss: 0.0511
Epoch 9/50
5683/5683 ━━━━━━━━━━━━━━━━━━━━ 2s 352us/step - loss: 0.0187 - val_loss: 0.0441
Epoch 10/50
5683/5683 ━━━━━━━━━━━━━━━━━━━━ 2s 355us/step - loss: 0.0179 - val_loss: 0.0233
Epoch 11/50
5683/5683 ━━━━━━━━━━━━━━━━━━━━ 2s 363us/step - loss: 0.0178 - val_